In [9]:
import json
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

# ============================================================
# 1. FILE PATHS
# ============================================================

INPUT_JSON = "gov_data_be_sampled_300.json"

OUTPUT_XLSX = "gov_data_be_300_annotation.xlsx"
OUTPUT_CSV = "gov_data_be_300_annotation.csv"


# ============================================================
# 2. LOAD JSON
# ============================================================

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Records loaded: {len(records)}")

# Make sure we really have 300 records
assert len(records) == 300, (
    f"Expected 300 records, but found {len(records)}"
)


# ============================================================
# 3. HELPER FUNCTION
# ============================================================

def list_to_text(value):
    """
    Convert JSON lists into readable text.
    Example:
        ["health", "government", "data"]
    becomes:
        "health; government; data"
    """
    if isinstance(value, list):
        return "; ".join(str(x) for x in value)

    if value is None:
        return ""

    return str(value)


# ============================================================
# 4. CREATE CLEAN DATAFRAME
# ============================================================

rows = []

for annotation_id, record in enumerate(records, start=1):

    rows.append({
        # ----------------------------------------------------
        # Source information
        # ----------------------------------------------------

        "Annotation ID": annotation_id,

        "Dataset ID": record.get("id", ""),

        "Domains": list_to_text(
            record.get("domains", "")
        ),

        "Title": record.get("title", ""),

        "Description": record.get("description", ""),

        "Keywords": list_to_text(
            record.get("keywords", "")
        ),

        # ----------------------------------------------------
        # Manual annotation fields
        # ----------------------------------------------------

        "Title Concepts": "",

        "Description Concepts": "",

        "Title Completeness": "",

        "Description Completeness": "",

        "Title Relevance": "",

        "Description Relevance": "",
    })


df = pd.DataFrame(rows)


# ============================================================
# 5. VERIFY DATAFRAME
# ============================================================

expected_columns = [
    "Annotation ID",
    "Dataset ID",
    "Domains",
    "Title",
    "Description",
    "Keywords",
    "Title Concepts",
    "Description Concepts",
    "Title Completeness",
    "Description Completeness",
    "Title Relevance",
    "Description Relevance",
]

assert list(df.columns) == expected_columns
assert len(df) == 300

print("\nDataFrame created successfully.")
print("Shape:", df.shape)

print("\nColumns:")
for number, column in enumerate(df.columns, start=1):
    print(f"{number}. {column}")


# ============================================================
# 6. SAVE CSV
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nCSV saved as: {OUTPUT_CSV}")


# ============================================================
# 7. SAVE REAL EXCEL FILE
# ============================================================

df.to_excel(
    OUTPUT_XLSX,
    index=False,
    sheet_name="Annotations"
)

print(f"Excel saved as: {OUTPUT_XLSX}")


# ============================================================
# 8. FORMAT THE EXCEL WORKBOOK
# ============================================================

wb = load_workbook(OUTPUT_XLSX)

ws = wb["Annotations"]


# ------------------------------------------------------------
# Freeze header row and source columns
# ------------------------------------------------------------

# Columns A-C remain visible while scrolling horizontally
ws.freeze_panes = "D2"


# ------------------------------------------------------------
# Add filters
# ------------------------------------------------------------

ws.auto_filter.ref = ws.dimensions


# ------------------------------------------------------------
# Header formatting
# ------------------------------------------------------------

header_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7"
)

header_font = Font(
    bold=True
)

thin_border = Border(
    bottom=Side(
        style="thin",
        color="808080"
    )
)

for cell in ws[1]:

    cell.font = header_font
    cell.fill = header_fill

    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True
    )

    cell.border = thin_border


# ------------------------------------------------------------
# Format all data cells
# ------------------------------------------------------------

for row in ws.iter_rows(
    min_row=2,
    max_row=ws.max_row
):

    for cell in row:

        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )


# ============================================================
# 9. COLUMN WIDTHS
# ============================================================

column_widths = {

    "A": 16,    # Annotation ID
    "B": 45,    # Dataset ID
    "C": 35,    # Domains
    "D": 55,    # Title
    "E": 90,    # Description
    "F": 45,    # Keywords

    "G": 40,    # Title Concepts
    "H": 50,    # Description Concepts

    "I": 25,    # Title Completeness
    "J": 30,    # Description Completeness

    "K": 20,    # Title Relevance
    "L": 25,    # Description Relevance
}

for column, width in column_widths.items():

    ws.column_dimensions[column].width = width


# ============================================================
# 10. HEADER HEIGHT
# ============================================================

ws.row_dimensions[1].height = 40


# ============================================================
# 11. SET VIEW
# ============================================================

ws.sheet_view.showGridLines = True


# ============================================================
# 12. SAVE FINAL WORKBOOK
# ============================================================

wb.save(OUTPUT_XLSX)

print("\n======================================")
print("DONE")
print("======================================")
print(f"Excel: {OUTPUT_XLSX}")
print(f"CSV:   {OUTPUT_CSV}")
print(f"Records: {len(df)}")
print(f"Columns: {len(df.columns)}")

Records loaded: 300

DataFrame created successfully.
Shape: (300, 12)

Columns:
1. Annotation ID
2. Dataset ID
3. Domains
4. Title
5. Description
6. Keywords
7. Title Concepts
8. Description Concepts
9. Title Completeness
10. Description Completeness
11. Title Relevance
12. Description Relevance

CSV saved as: gov_data_be_300_annotation.csv
Excel saved as: gov_data_be_300_annotation.xlsx

DONE
Excel: gov_data_be_300_annotation.xlsx
CSV:   gov_data_be_300_annotation.csv
Records: 300
Columns: 12
